# 26 — Language Models & Text Generation

**Learning objective.** Understand next-token probability, perplexity and decoding using a transparent bigram language model.

This notebook follows the track contract: concept → inspectable implementation → rendered result → failure modes → production implication.

## Mental model

**prompt/context → next-token distribution → decoding policy → generated continuation**

Follow the information transformation first; treat the API as an implementation detail.

### Reference visual

<p align="center"><img src="https://commons.wikimedia.org/wiki/Special:Redirect/file/Full%20GPT%20architecture.svg" width="560" alt="GPT architecture"/></p>

<sub>Wikimedia Commons — “Full GPT architecture.svg”, CC0 1.0. Attribution details: `VISUAL_REFERENCES.md`.</sub>

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Lower **temperature** | distribution sharpens | generation becomes more deterministic/conservative |
| Restrict **top-k/top-p** | low-probability tokens are removed | diversity and tail-risk change |
| Use greedy vs beam vs sampling | search policy changes | output quality/diversity/repetition change |

> Write down what should move downstream before changing a control.

## Think before running the next cell

1. At temperature approaching zero, what should happen to sampling?
2. Can lower perplexity guarantee factual answers?

### When to use
Use generation when open-ended composition is part of the task.

### When not to use / caution
Do not equate high-probability continuation with truth, grounding or safety.

### Debugging lens
Inspect token probabilities and decoding policy separately from model weights; many output changes come from decoding alone.

In [1]:
from pathlib import Path
import re, random, math, json
import numpy as np
import pandas as pd
np.random.seed(42); random.seed(42)
print("Reproducibility seed: 42")

Reproducibility seed: 42


A language model estimates a probability distribution over the next token:

$$P(w_1,\dots,w_T)=\prod_{t=1}^{T}P(w_t\mid w_{<t})$$

Modern causal transformers condition on the entire previous context. A bigram model conditions only on the previous token, making the mechanics easy to inspect.

In [2]:
from collections import defaultdict, Counter
corpus=[
 'nlp models learn patterns from language',
 'language models predict the next token',
 'deep models learn contextual representations',
 'nlp systems turn language into useful predictions']
counts=defaultdict(Counter); vocab={'<EOS>','<UNK>'}
for s in corpus:
    toks=['<BOS>']+s.split()+['<EOS>']; vocab.update(toks)
    for a,b in zip(toks,toks[1:]): counts[a][b]+=1

def probs(prev,alpha=0.2):
    candidates=sorted(vocab-{'<BOS>'}); total=sum(counts[prev].values())+alpha*len(candidates)
    return {w:(counts[prev][w]+alpha)/total for w in candidates}
print('P(next | language) top 6:')
print(sorted(probs('language').items(),key=lambda x:-x[1])[:6])

P(next | language) top 6:
[('<EOS>', 0.17142857142857143), ('into', 0.17142857142857143), ('models', 0.17142857142857143), ('<UNK>', 0.028571428571428574), ('contextual', 0.028571428571428574), ('deep', 0.028571428571428574)]


In [3]:
def generate(max_tokens=12):
    out=[]; prev='<BOS>'
    for _ in range(max_tokens):
        p=probs(prev); nxt=max(p,key=p.get)   # greedy decoding
        if nxt=='<EOS>': break
        out.append(nxt); prev=nxt
    return ' '.join(out)
print('greedy generation:',generate())

def perplexity(sentence):
    toks=['<BOS>']+sentence.split()+['<EOS>']; logp=0
    for a,b in zip(toks,toks[1:]):
        a = a if a in vocab else '<UNK>'; b = b if b in vocab else '<UNK>'
        logp += math.log(probs(a)[b])
    return math.exp(-logp/(len(toks)-1))
for s in ['language models predict the next token','quantum bananas drive language']:
    print(f'{s!r} -> perplexity {perplexity(s):.2f}')

greedy generation: nlp models learn contextual representations
'language models predict the next token' -> perplexity 4.91
'quantum bananas drive language' -> perplexity 17.96


### Decoding choices
- **Greedy:** highest-probability token; deterministic but can be repetitive.
- **Beam search:** keeps several high-probability partial sequences.
- **Top-k / nucleus sampling:** samples from a restricted probability mass; useful for diverse generation.
- **Temperature:** reshapes the distribution before sampling.

Perplexity measures how surprised a language model is by reference text; it is not a direct measure of factuality, usefulness or safety.

---
## Production takeaways
- Treat decoding, privacy, robustness and task heads as explicit system design choices.
- Keep evaluation aligned with the actual task and deployment risk.